# MNIST 卷積神經網路 (CNN) 實作與分析

本 Notebook 依題目 1-1 ~ 1-4 的順序進行,使用 **TensorFlow / Keras** 高階 API。

## 整體流程總覽
| 步驟 | 內容 | 對應題號 |
|---|---|---|
| 0 | 環境設定、載入 MNIST、切分 55000 / 5000 / 10000 | 前置 |
| 1 | 設計 CNN 架構、自訂訓練迴圈(每個 iteration 記錄 loss / accuracy) | 1-1 |
| 2 | 畫學習曲線、Train/Val/Test accuracy、權重與偏差直方圖 | 1-1 |
| 3 | 比較不同 **stride** 與 **filter size**(3×2 實驗) | 1-1 |
| 4 | 正確 / 錯誤分類範例、混淆矩陣、錯誤分析 | 1-2 |
| 5 | 觀察 conv1 / conv2 feature map,討論隨深度的變化 | 1-3 |
| 6 | 加入 L2 regularization,比較不同 α 並討論 | 1-4 |

> 註:題目提供的 `tensorflow.examples.tutorials.mnist` 已被移除,且 `tf.keras.dataset` 拼字應為 `tf.keras.datasets`,本 Notebook 已修正。

## 0. 環境設定與資料載入

In [ ]:
import time, random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
(x_all, y_all), (x_test, y_test) = keras.datasets.mnist.load_data()

# 60000 張訓練資料 -> 55000 train + 5000 validation (與題目一致)
x_train, y_train = x_all[:55000], y_all[:55000]
x_val,   y_val   = x_all[55000:], y_all[55000:]

def prep(x):
    # 正規化到 [0,1],並加上 channel 維度 -> (N, 28, 28, 1)
    return (x.astype("float32") / 255.0)[..., np.newaxis]

x_train, x_val, x_test = prep(x_train), prep(x_val), prep(x_test)
print("train:", x_train.shape, "val:", x_val.shape, "test:", x_test.shape)

In [ ]:
# 對應 Fig.1:顯示資料集範例
fig, axes = plt.subplots(6, 12, figsize=(12, 6))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(x_train[i, :, :, 0], cmap="gray"); ax.axis("off")
plt.suptitle("Figure 1: Examples of handwritten digits"); plt.tight_layout(); plt.show()

## 1-1 設計 CNN 架構與訓練

### 架構設計
```
Input (28×28×1)
 → Conv1 (32 filters, k×k, stride s, same, ReLU) → MaxPool 2×2
 → Conv2 (64 filters, k×k, stride s, same, ReLU) → MaxPool 2×2
 → Flatten → Dense1 (128, ReLU) → Output (10, logits → softmax)
```
- `build_cnn(filter_size, stride, l2_alpha)` 讓 filter size / stride / L2 可調,方便後續實驗。
- 損失函數為 cross entropy,優化器 Adam。
- **自訂訓練迴圈**:Fig.2 的 x 軸為 iteration(每次一個 mini-batch),所以我們每個 iteration 紀錄 cross entropy,並每隔 `eval_every` 個 iteration 評估 train(目前 batch)/ validation / test accuracy。

In [ ]:
def build_cnn(filter_size=3, stride=1, l2_alpha=0.0, c1=32, c2=64, dense_units=128):
    reg = regularizers.l2(l2_alpha) if l2_alpha > 0 else None   # Keras: alpha * sum(w^2)
    inp = keras.Input(shape=(28, 28, 1))
    x = layers.Conv2D(c1, filter_size, strides=stride, padding="same", activation="relu",
                      kernel_regularizer=reg, name="conv1")(inp)
    x = layers.MaxPooling2D(2, name="pool1")(x)
    x = layers.Conv2D(c2, filter_size, strides=stride, padding="same", activation="relu",
                      kernel_regularizer=reg, name="conv2")(x)
    x = layers.MaxPooling2D(2, name="pool2")(x)
    x = layers.Flatten()(x)
    x = layers.Dense(dense_units, activation="relu", kernel_regularizer=reg, name="dense1")(x)
    out = layers.Dense(10, kernel_regularizer=reg, name="output")(x)   # logits
    return keras.Model(inp, out)

build_cnn().summary()

In [ ]:
def evaluate_acc(model, x, y, bs=2000):
    correct = 0
    for i in range(0, len(x), bs):
        logits = model(x[i:i+bs], training=False)
        correct += np.sum(np.argmax(logits.numpy(), 1) == y[i:i+bs])
    return correct / len(x)

def train_model(model, iters=300, batch_size=128, lr=1e-3, eval_every=5, verbose=True):
    opt = keras.optimizers.Adam(lr)
    ce_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

    @tf.function
    def step(xb, yb):
        with tf.GradientTape() as tape:
            logits = model(xb, training=True)
            ce = ce_fn(yb, logits)
            reg = tf.add_n(model.losses) if model.losses else 0.0
            total = ce + reg
        grads = tape.gradient(total, model.trainable_variables)
        opt.apply_gradients(zip(grads, model.trainable_variables))
        acc = tf.reduce_mean(tf.cast(tf.argmax(logits, 1) == tf.cast(yb, tf.int64), tf.float32))
        return ce, total, acc

    hist = dict(ce=[], total=[], eval_iter=[], train_acc=[], val_acc=[], test_acc=[])
    n = len(x_train); perm = np.random.permutation(n); ptr = 0
    t0 = time.time()
    for it in range(iters + 1):
        if ptr + batch_size > n:
            perm = np.random.permutation(n); ptr = 0
        idx = perm[ptr:ptr + batch_size]; ptr += batch_size
        ce, total, acc = step(x_train[idx], y_train[idx])
        hist["ce"].append(float(ce)); hist["total"].append(float(total))
        if it % eval_every == 0:
            hist["eval_iter"].append(it)
            hist["train_acc"].append(float(acc))
            hist["val_acc"].append(evaluate_acc(model, x_val, y_val))
            hist["test_acc"].append(evaluate_acc(model, x_test, y_test))
            if verbose and it % (eval_every * 10) == 0:
                print(f"iter {it:4d} | CE {float(ce):.4f} | train {float(acc):.4f} | "
                      f"val {hist['val_acc'][-1]:.4f} | test {hist['test_acc'][-1]:.4f}")
    hist["time"] = time.time() - t0
    return hist

### 訓練基準模型(filter 3×3, stride 1)

In [ ]:
BASE_K, BASE_S = 3, 1
base_model = build_cnn(BASE_K, BASE_S)
base_hist = train_model(base_model, iters=300, batch_size=128, eval_every=5)
print("Final test acc:", base_hist["test_acc"][-1])

### 畫圖:學習曲線、accuracy、權重 / 偏差分布 (對應 Fig.2)

In [ ]:
LAYERS = ["conv1", "conv2", "dense1", "output"]

def get_params(model, kind="kernel"):
    out = {}
    for name in LAYERS:
        w, b = model.get_layer(name).get_weights()
        out[name] = (w if kind == "kernel" else b).ravel()
    return out

def plot_results(hist, model, title="", xlim=None, bins=100):
    fig, axes = plt.subplots(3, 2, figsize=(11, 12))
    ax = axes[0, 0]
    ax.plot(hist["eval_iter"], hist["train_acc"], "b", label="Training Accuracy")
    ax.plot(hist["eval_iter"], hist["val_acc"],   "r", label="Validation Accuracy")
    ax.plot(hist["eval_iter"], hist["test_acc"],  "g", label="Test Accuracy")
    ax.set_title("Accuracy"); ax.set_xlabel("Iteration"); ax.set_ylabel("Accuracy rate")
    ax.set_ylim(0.9, 1.005); ax.legend(loc="lower right")

    ax = axes[0, 1]
    ax.plot(hist["ce"], "b", label="Cross entropy")
    ax.set_title("Learning Curve"); ax.set_xlabel("Iteration"); ax.set_ylabel("Loss"); ax.legend()

    ws = get_params(model, "kernel")
    for ax, name in zip(axes[1:].ravel(), LAYERS):
        ax.hist(ws[name], bins=bins, range=xlim)
        ax.set_title(f"Histogram of {name}"); ax.set_xlabel("Value"); ax.set_ylabel("Number")
    plt.suptitle(title); plt.tight_layout(); plt.show()

def plot_bias_hist(model, bins=50):
    bs = get_params(model, "bias")
    fig, axes = plt.subplots(1, 4, figsize=(16, 3))
    for ax, name in zip(axes, LAYERS):
        ax.hist(bs[name], bins=bins); ax.set_title(f"Bias of {name}"); ax.set_xlabel("Value")
    plt.tight_layout(); plt.show()

plot_results(base_hist, base_model, title=f"Baseline CNN (k={BASE_K}, stride={BASE_S})")
plot_bias_hist(base_model)

## 1-1 (續) 分析不同 stride 與 filter size 的影響

實驗設計:filter size ∈ {3, 5, 7} × stride ∈ {1, 2},共 6 組,其他超參數(優化器、batch size、iterations、seed)全部相同。
記錄:最終 train / val / test accuracy、參數量、訓練時間。

In [ ]:
filter_sizes = [3, 5, 7]
strides = [1, 2]
exp = {}
for s in strides:
    for k in filter_sizes:
        tf.random.set_seed(SEED); np.random.seed(SEED)
        m = build_cnn(k, s)
        h = train_model(m, iters=300, eval_every=10, verbose=False)
        exp[(k, s)] = dict(params=m.count_params(), hist=h)
        print(f"k={k}, stride={s} | params={m.count_params():>8,} | "
              f"test={h['test_acc'][-1]:.4f} | val={h['val_acc'][-1]:.4f} | time={h['time']:.1f}s")

In [ ]:
# 結果表格
print(f"{'filter':>6} {'stride':>6} {'params':>10} {'val_acc':>8} {'test_acc':>9} {'time(s)':>8}")
for (k, s), r in exp.items():
    h = r["hist"]
    print(f"{k:>6} {s:>6} {r['params']:>10,} {h['val_acc'][-1]:>8.4f} {h['test_acc'][-1]:>9.4f} {h['time']:>8.1f}")

# 視覺化:test accuracy 曲線 + 最終數值長條圖
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for (k, s), r in exp.items():
    axes[0].plot(r["hist"]["eval_iter"], r["hist"]["test_acc"], label=f"k={k}, s={s}")
axes[0].set_ylim(0.95, 1.0); axes[0].set_xlabel("Iteration"); axes[0].set_ylabel("Test accuracy")
axes[0].set_title("Test accuracy vs. iteration"); axes[0].legend()

labels = [f"k{k}/s{s}" for (k, s) in exp]
axes[1].bar(labels, [r["hist"]["test_acc"][-1] for r in exp.values()])
axes[1].set_ylim(0.97, 1.0); axes[1].set_title("Final test accuracy"); axes[1].set_ylabel("Accuracy")
plt.tight_layout(); plt.show()

### 📝 分析(請依你實際跑出的數字撰寫)
可從以下面向討論,典型觀察如下,**請以自己的結果為準**:

**Stride 的影響**
- stride=2 讓 feature map 邊長減半 → 後續 Flatten 後的維度與 Dense 參數量大幅下降,訓練速度較快。
- 但 stride 變大會跳過部分像素,損失空間細節(MNIST 筆劃細,影響更明顯),accuracy 通常略低於 stride=1。

**Filter size 的影響**
- filter 越大,感受野(receptive field)越大,一次能看到更大範圍的筆劃結構,但參數量與計算量成平方成長(3×3=9, 5×5=25, 7×7=49)。
- MNIST 圖片小(28×28),3×3~5×5 通常已足夠;7×7 容易增加過擬合風險,收益有限。
- 在 stride=2 時,太大的 filter 搭配 padding 可補償部分資訊損失。

**結論範例**:在準確率、參數量、訓練時間的取捨下,選擇 ____ 作為最終架構,理由:____。

## 1-2 正確 / 錯誤分類範例 (對應 Fig.3)
使用基準模型對 10000 張測試圖預測,挑選正確與錯誤的範例,並用混淆矩陣找出最常混淆的數字組合。

In [ ]:
def predict(model, x, bs=2000):
    out = [model(x[i:i+bs], training=False).numpy() for i in range(0, len(x), bs)]
    return np.concatenate(out)

logits = predict(base_model, x_test)
probs = tf.nn.softmax(logits).numpy()
pred = probs.argmax(1)
correct_idx = np.where(pred == y_test)[0]
wrong_idx   = np.where(pred != y_test)[0]
print(f"Test accuracy: {len(correct_idx)/len(y_test):.4f} | misclassified: {len(wrong_idx)}")

def show_samples(idxs, title, n=10):
    fig, axes = plt.subplots(2, n // 2, figsize=(2 * n // 2, 5.2))
    for ax, i in zip(axes.ravel(), idxs[:n]):
        ax.imshow(x_test[i, :, :, 0], cmap="gray")
        ax.set_title(f"label: {y_test[i]}, pred: {pred[i]}\nconf: {probs[i, pred[i]]:.2f}", fontsize=9)
        ax.axis("off")
    plt.suptitle(title); plt.tight_layout(); plt.show()

rng = np.random.RandomState(0)
show_samples(rng.choice(correct_idx, 10, replace=False), "Correctly classified")
show_samples(wrong_idx, "Misclassified")

In [ ]:
# 混淆矩陣
cm = tf.math.confusion_matrix(y_test, pred, num_classes=10).numpy()
plt.figure(figsize=(6, 5)); plt.imshow(cm, cmap="Blues"); plt.colorbar()
for i in range(10):
    for j in range(10):
        plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=7,
                 color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.xticks(range(10)); plt.yticks(range(10)); plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Confusion matrix"); plt.show()

# 最常見的錯誤組合
off = cm.copy(); np.fill_diagonal(off, 0)
pairs = sorted([(off[i, j], i, j) for i in range(10) for j in range(10) if off[i, j] > 0], reverse=True)[:8]
print("最常見的錯誤 (true -> pred : 次數)")
for c, i, j in pairs: print(f"  {i} -> {j} : {c}")

# 錯誤樣本的信心分布 vs 正確樣本
plt.figure(figsize=(6, 3.5))
plt.hist(probs[correct_idx].max(1), bins=30, alpha=.6, label="correct", density=True)
plt.hist(probs[wrong_idx].max(1),   bins=30, alpha=.6, label="wrong",   density=True)
plt.xlabel("Max softmax probability"); plt.legend(); plt.title("Confidence"); plt.show()

### 📝 討論(請依你實際結果補充)
建議討論重點:
1. **整體表現**:測試 accuracy 約 99% 左右,代表僅約 100 張左右誤判。
2. **常見混淆組合**:例如 4↔9、3↔5、7↔2、8↔3 等外形相近的數字,混淆矩陣上離對角線最大的格子即為證據。
3. **誤判原因**:
   - 書寫風格特殊(如 Fig.3 左的 9 形似 4,頂部沒有閉合)
   - 筆劃斷裂、多餘雜點(如 Fig.3 右圖旁邊的小點)
   - 部分樣本連人眼也難以判斷,屬於資料標註 / 本身模糊的問題
4. **信心分布**:誤判樣本的最大機率通常較低,可考慮用信心門檻把低信心樣本交由人工複核。

## 1-3 Feature map 觀察 (對應 Fig.4)
承 1-2,挑選一張**誤判**與一張**正確**的樣本,取出 conv1(28×28)與 conv2(14×14)的 feature map,觀察隨深度加深的變化。

In [ ]:
feat_model = keras.Model(base_model.input,
                         [base_model.get_layer("conv1").output, base_model.get_layer("conv2").output])

def show_feature_maps(i, n_ch=4):
    f1, f2 = [f.numpy()[0] for f in feat_model(x_test[i:i+1], training=False)]
    fig, axes = plt.subplots(2, n_ch + 1, figsize=(2.6 * (n_ch + 1), 5.4))
    axes[0, 0].imshow(x_test[i, :, :, 0], cmap="gray")
    axes[0, 0].set_title(f"label: {y_test[i]}, pred: {pred[i]}")
    axes[1, 0].axis("off")
    for c in range(n_ch):
        axes[0, c + 1].imshow(f1[:, :, c], cmap="gray"); axes[0, c + 1].set_title(f"conv1 ch{c} {f1.shape[:2]}")
        axes[1, c + 1].imshow(f2[:, :, c], cmap="gray"); axes[1, c + 1].set_title(f"conv2 ch{c} {f2.shape[:2]}")
    for ax in axes.ravel(): ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.show()

show_feature_maps(wrong_idx[0])                       # 誤判樣本
show_feature_maps(rng.choice(correct_idx))            # 正確樣本

### 📝 描述:feature map 隨深度的變化
典型觀察(請對照你自己的圖):
- **conv1(淺層)**:feature map 與原圖外形幾乎一致,解析度高,主要是在偵測**低階特徵**——邊緣、特定方向的線段、筆劃明暗,可明顯看出原始數字輪廓。
- **conv2(較深層)**:經過 pooling 解析度下降(28→14),啟動區域變得**稀疏且局部**,不再像原圖,而是對應到「弧線、轉角、交叉、封閉環」等**較高階、較抽象的部分特徵**。
- 深度越深,**空間細節越少、語意性越強**,感受野更大,資訊從「像素是什麼」轉為「出現了哪些結構」,這些組合最後交給全連接層做分類。
- 誤判樣本與正確樣本的 conv2 啟動模式不同:誤判的 9 缺少閉合的環,啟動型態更像 4 的「開口 + 橫線」,可解釋為什麼被判成 4。

## 1-4 加入 L2 Regularization (對應 Fig.5)

$$E = -\frac{1}{N}\sum_{n}\sum_{k} y_{nk}\ln t_{nk} + \alpha\lVert\omega\rVert_2^2$$

在 Keras 中對每一層 kernel 加上 `regularizers.l2(α)`,其懲罰項正好為 α·Σw²(與公式一致;偏差 b 不正則化)。訓練迴圈中 `total = CE + Σ reg`,**學習曲線畫的是純 cross entropy**,另外也記錄含正則項的 total loss。
先掃描不同 α,再挑一個明顯的 α 畫完整圖表(對應 Fig.5 權重集中於 0)。

In [ ]:
alphas = [0.0, 1e-4, 1e-3, 1e-2, 1e-1]
l2_res = {}
for a in alphas:
    tf.random.set_seed(SEED); np.random.seed(SEED)
    m = build_cnn(BASE_K, BASE_S, l2_alpha=a)
    h = train_model(m, iters=300, eval_every=5, verbose=False)
    l2_res[a] = dict(model=m, hist=h)
    w = np.concatenate(list(get_params(m, "kernel").values()))
    print(f"alpha={a:<7} | train(batch)={h['train_acc'][-1]:.4f} | val={h['val_acc'][-1]:.4f} | "
          f"test={h['test_acc'][-1]:.4f} | |w|<0.01 比例={np.mean(np.abs(w)<0.01):.3f} | mean|w|={np.mean(np.abs(w)):.4f}")

In [ ]:
# 比較不同 alpha 的 test / val accuracy、train-test gap、total loss
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for a, r in l2_res.items():
    h = r["hist"]
    axes[0].plot(h["eval_iter"], h["test_acc"], label=f"α={a}")
    axes[1].plot([x - y for x, y in zip(h["train_acc"], h["test_acc"])], label=f"α={a}")
    axes[2].plot(h["total"], label=f"α={a}")
axes[0].set_ylim(0.95, 1.0); axes[0].set_title("Test accuracy"); axes[0].legend()
axes[1].set_title("Train(batch) acc − Test acc (generalization gap)")
axes[2].set_yscale("log"); axes[2].set_title("Total loss (CE + L2)")
for ax in axes: ax.set_xlabel("Iteration (x5 for first two)" if ax is not axes[2] else "Iteration")
plt.tight_layout(); plt.show()

In [ ]:
# 選一個 alpha 畫完整圖(對應 Fig.5):權重集中在 0 附近,x 軸縮放到 [-0.1, 0.1]
ALPHA_SHOW = 1e-2
plot_results(l2_res[ALPHA_SHOW]["hist"], l2_res[ALPHA_SHOW]["model"],
             title=f"CNN with L2 regularization (alpha={ALPHA_SHOW})", xlim=(-0.1, 0.1), bins=50)

# 同樣縮放範圍下的無正則化版本,方便並排比較
plot_results(l2_res[0.0]["hist"], l2_res[0.0]["model"],
             title="CNN without regularization (same x-range)", xlim=(-0.1, 0.1), bins=50)

### 📝 討論 L2 regularization 的效果(請依實際結果補充)
- **權重分布**:加入 L2 後,權重被拉向 0,直方圖在 0 附近形成尖峰(Fig.5 的現象);α 越大越集中。大量權重趨近 0 代表模型有效容量被壓縮,只保留對 loss 貢獻明顯的連結。
- **Loss**:total loss 起始值變大(含 α‖ω‖² 項),學習曲線在初期下降更陡。α 過大時,正則項主導,CE 無法降到很低。
- **Accuracy 與泛化**:
  - 適當的 α(如 1e-4~1e-3)可縮小 train–test 差距、使 validation/test 曲線更穩定,有時微幅提升 test accuracy。
  - α 過大(如 1e-1)會造成 underfitting,train 與 test accuracy 都下降。
  - MNIST 本身很簡單,無正則化的 CNN 已接近 99%,所以 L2 的 accuracy 改善幅度通常有限,主要價值在於權重穩定與抑制過擬合。
- **結論**:選擇 α = ____ 作為折衷,理由:____。

## 總結與建議報告架構
1. 資料集與前處理(55000/5000/10000)
2. 網路架構與超參數
3. 1-1:學習曲線、accuracy、權重/偏差直方圖;stride × filter size 比較表與分析
4. 1-2:正確/錯誤範例、混淆矩陣、誤判原因
5. 1-3:feature map 深度比較
6. 1-4:L2 的 α 掃描與討論
7. 結論

> 提醒:每次重新執行,數值會因隨機性略有不同;所有 📝 區塊請以你自己的實驗結果改寫。